# End-to-end symbolic analysis pipeline

Processed PPG pipeline for one CSV per session:

`segmented windows → processed-stationarity filter → peak detection → uncorrected PPI QC → frozen Guzzetti symbolic dynamics → validated session CSV`

Public entry point: `symbolic_analysis_run(session_id)`.

In [2]:
from numbers import Integral
from pathlib import Path
import logging
import sys

import numpy as np
import pandas as pd
from scipy.signal import find_peaks


def locate_repo_root(start: Path) -> Path:
    """Locate the repository from root or the notebook directory."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "phase1" / "src").is_dir():
            return candidate
    raise FileNotFoundError("Cannot locate repository root.")


REPO_ROOT = locate_repo_root(Path.cwd())
PHASE1_ROOT = REPO_ROOT / "phase1"
SRC_DIR = PHASE1_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from dataloader.loader import load_segmented_session
from symbolic import analyze_symbolic_dynamics

WINDOW_SIZES = (60, 120, 180)
REPRESENTATION = "processed"
STATE_NAMES = {0: "Awake", 1: "Drowsy"}

SCHEMA_VERSION = "1.0.0"
ANALYSIS_ID = "symbolic_processed_v1"

SYMBOLIC_N_LEVELS = 6
SYMBOLIC_WORD_LENGTH = 3

MIN_PEAK_DISTANCE_S = 0.30
PROMINENCE_FRACTION = 0.20
PPI_MIN_S = 0.30
PPI_MAX_S = 2.00
ABRUPT_CHANGE_THRESHOLD = 0.20
MIN_PEAK_COVERAGE_PCT = 90.0

SEGMENTED_DATA_DIR = PHASE1_ROOT / "segmentated_data" / "dhdata"
OUTPUT_DIR = PHASE1_ROOT / "results" / "symbolic" / "processed"

LOGGER = logging.getLogger("symbolic_analysis")
if not LOGGER.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    LOGGER.addHandler(handler)
LOGGER.setLevel(logging.INFO)
LOGGER.propagate = False

## Frozen output schema and policies

The output universe is every processed window that passed processed-stationarity. A row is retained even if downstream PPI/symbolic validation fails.

`analysis_included` requires `ppi_qc_pass AND symbolic_valid`; stationarity is already guaranteed by input filtering.

In [3]:
OUTPUT_COLUMNS = [
    "schema_version",
    "analysis_id",
    "session_id",
    "session",
    "source_file",
    "representation",
    "window_uid",
    "window_size_s",
    "window_id",
    "source_row_index",
    "state",
    "label_code",
    "start_time_s",
    "end_time_s",
    "sampling_rate_hz",
    "n_samples",
    "stationarity_score_processed",
    "stationarity_pass_processed",
    "symbolic_n_levels",
    "symbolic_word_length",
    "peak_min_distance_s",
    "peak_prominence_fraction",
    "ppi_min_s",
    "ppi_max_s",
    "abrupt_change_threshold",
    "min_peak_coverage_pct",
    "prominence_threshold",
    "min_required_intervals",
    "n_peaks",
    "peak_coverage_pct",
    "n_intervals",
    "n_words",
    "mean_hr_bpm",
    "median_hr_bpm",
    "mean_ppi_ms",
    "median_ppi_ms",
    "sd_ppi_ms",
    "min_ppi_ms",
    "max_ppi_ms",
    "ppi_cv",
    "n_invalid_ppi",
    "invalid_ppi_pct",
    "n_abrupt_gt_20pct",
    "abrupt_gt_20pct",
    "ppi_qc_pass",
    "pct_0v",
    "pct_1v",
    "pct_2lv",
    "pct_2uv",
    "pct_2v",
    "symbolic_invariants_pass",
    "symbolic_valid",
    "analysis_included",
    "failure_stage",
    "invalid_reason",
]

NUMERIC_RESULT_COLUMNS = [
    "prominence_threshold",
    "min_required_intervals",
    "n_peaks",
    "peak_coverage_pct",
    "n_intervals",
    "n_words",
    "mean_hr_bpm",
    "median_hr_bpm",
    "mean_ppi_ms",
    "median_ppi_ms",
    "sd_ppi_ms",
    "min_ppi_ms",
    "max_ppi_ms",
    "ppi_cv",
    "n_invalid_ppi",
    "invalid_ppi_pct",
    "n_abrupt_gt_20pct",
    "abrupt_gt_20pct",
    "pct_0v",
    "pct_1v",
    "pct_2lv",
    "pct_2uv",
    "pct_2v",
]

assert ABRUPT_CHANGE_THRESHOLD == 0.20

## Peak detection and PPI extraction

The detector is the Session 1 pilot configuration: no extra smoothing, 0.30-s minimum distance and prominence equal to 20% of each window's P95−P5 robust range.

Abrupt changes are reported but are not an automatic exclusion criterion.

In [4]:
def detect_peaks_and_ppi(
    signal: np.ndarray,
    sampling_rate_hz: float,
) -> dict:
    """Detect systolic maxima and return uncorrected PPI diagnostics."""
    values = np.asarray(signal, dtype=np.float64)
    if values.ndim != 1 or values.size == 0:
        raise ValueError("Processed PPG must be a non-empty 1D array.")
    if not np.isfinite(values).all():
        raise ValueError("Processed PPG contains non-finite values.")

    robust_range = float(
        np.percentile(values, 95) - np.percentile(values, 5)
    )
    if not np.isfinite(robust_range) or robust_range <= 0:
        raise ValueError("Processed PPG has no positive robust range.")

    prominence_threshold = PROMINENCE_FRACTION * robust_range
    peaks, properties = find_peaks(
        values,
        distance=int(np.ceil(MIN_PEAK_DISTANCE_S * sampling_rate_hz)),
        prominence=prominence_threshold,
    )
    if peaks.size < 2:
        raise ValueError("Fewer than two peaks were detected.")

    peak_times_s = peaks.astype(np.float64) / sampling_rate_hz
    ppi_s = np.diff(peak_times_s)
    relative_change = (
        np.abs(np.diff(ppi_s)) / ppi_s[:-1]
        if ppi_s.size >= 2
        else np.empty(0, dtype=np.float64)
    )

    return {
        "peaks": peaks,
        "peak_times_s": peak_times_s,
        "ppi_s": ppi_s,
        "prominences": properties["prominences"],
        "prominence_threshold": prominence_threshold,
        "relative_change": relative_change,
    }

## Window-level analysis

PPI QC requires finite intervals, zero physiologically invalid intervals, sufficient interval count for the duration, and at least 90% peak-span coverage. Symbolic metrics are computed only after PPI QC passes.

In [5]:
def _base_result_row(window: dict, session_id: int) -> dict:
    """Create a complete output row before downstream analysis."""
    row = {
        "schema_version": SCHEMA_VERSION,
        "analysis_id": ANALYSIS_ID,
        "session_id": session_id,
        "session": f"session_{session_id:02d}",
        "source_file": f"sample_{session_id}.csv",
        "representation": REPRESENTATION,
        "window_uid": (
            f"session_{session_id:02d}__processed__"
            f"{window['window_size_s']}s__w{window['window_id']:03d}"
        ),
        "window_size_s": window["window_size_s"],
        "window_id": window["window_id"],
        "source_row_index": window["source_row_index"],
        "state": window["state"],
        "label_code": window["label_code"],
        "start_time_s": window["start_time_s"],
        "end_time_s": window["end_time_s"],
        "sampling_rate_hz": window["sampling_rate_hz"],
        "n_samples": window["signal"].size,
        "stationarity_score_processed": window["stationarity_score"],
        "stationarity_pass_processed": window["stationarity_pass"],
        "symbolic_n_levels": SYMBOLIC_N_LEVELS,
        "symbolic_word_length": SYMBOLIC_WORD_LENGTH,
        "peak_min_distance_s": MIN_PEAK_DISTANCE_S,
        "peak_prominence_fraction": PROMINENCE_FRACTION,
        "ppi_min_s": PPI_MIN_S,
        "ppi_max_s": PPI_MAX_S,
        "abrupt_change_threshold": ABRUPT_CHANGE_THRESHOLD,
        "min_peak_coverage_pct": MIN_PEAK_COVERAGE_PCT,
        "ppi_qc_pass": False,
        "symbolic_invariants_pass": False,
        "symbolic_valid": False,
        "analysis_included": False,
        "failure_stage": "",
        "invalid_reason": "",
    }
    row.update({column: np.nan for column in NUMERIC_RESULT_COLUMNS})
    return row


def analyze_stationary_window(window: dict, session_id: int) -> dict:
    """Analyze one stationary processed window without dropping failures."""
    row = _base_result_row(window, session_id)

    try:
        peak_result = detect_peaks_and_ppi(
            window["signal"],
            window["sampling_rate_hz"],
        )
    except (TypeError, ValueError, RuntimeError) as exc:
        row["failure_stage"] = "peak_detection"
        row["invalid_reason"] = str(exc)
        return row

    ppi_s = peak_result["ppi_s"]
    hr_bpm = 60.0 / ppi_s
    invalid_ppi = (
        ~np.isfinite(ppi_s)
        | (ppi_s < PPI_MIN_S)
        | (ppi_s > PPI_MAX_S)
    )
    relative_change = peak_result["relative_change"]

    duration_s = float(window["window_size_s"])
    peak_coverage_pct = (
        100.0
        * (peak_result["peak_times_s"][-1] - peak_result["peak_times_s"][0])
        / duration_s
    )
    min_required_intervals = max(
        SYMBOLIC_WORD_LENGTH,
        int(np.ceil(duration_s / PPI_MAX_S)) - 2,
    )

    n_intervals = int(ppi_s.size)
    n_invalid = int(invalid_ppi.sum())
    n_abrupt = int(
        np.sum(relative_change > ABRUPT_CHANGE_THRESHOLD)
    )

    row.update({
        "prominence_threshold": peak_result["prominence_threshold"],
        "min_required_intervals": min_required_intervals,
        "n_peaks": int(peak_result["peaks"].size),
        "peak_coverage_pct": peak_coverage_pct,
        "n_intervals": n_intervals,
        "mean_hr_bpm": float(np.mean(hr_bpm)),
        "median_hr_bpm": float(np.median(hr_bpm)),
        "mean_ppi_ms": float(1000.0 * np.mean(ppi_s)),
        "median_ppi_ms": float(1000.0 * np.median(ppi_s)),
        "sd_ppi_ms": (
            float(1000.0 * np.std(ppi_s, ddof=1))
            if n_intervals >= 2 else np.nan
        ),
        "min_ppi_ms": float(1000.0 * np.min(ppi_s)),
        "max_ppi_ms": float(1000.0 * np.max(ppi_s)),
        "ppi_cv": (
            float(np.std(ppi_s, ddof=1) / np.mean(ppi_s))
            if n_intervals >= 2 else np.nan
        ),
        "n_invalid_ppi": n_invalid,
        "invalid_ppi_pct": 100.0 * n_invalid / n_intervals,
        "n_abrupt_gt_20pct": n_abrupt,
        "abrupt_gt_20pct": (
            100.0 * n_abrupt / relative_change.size
            if relative_change.size else 0.0
        ),
    })

    qc_reasons = []
    if n_invalid:
        qc_reasons.append("physiologically_invalid_ppi")
    if n_intervals < min_required_intervals:
        qc_reasons.append("insufficient_intervals")
    if peak_coverage_pct < MIN_PEAK_COVERAGE_PCT:
        qc_reasons.append("insufficient_peak_coverage")

    finite_qc_values = np.asarray([
        row["mean_hr_bpm"],
        row["median_hr_bpm"],
        row["mean_ppi_ms"],
        row["median_ppi_ms"],
        row["min_ppi_ms"],
        row["max_ppi_ms"],
        row["peak_coverage_pct"],
    ])
    if not np.isfinite(finite_qc_values).all():
        qc_reasons.append("nonfinite_ppi_metrics")

    row["ppi_qc_pass"] = not qc_reasons
    if qc_reasons:
        row["failure_stage"] = "ppi_qc"
        row["invalid_reason"] = "|".join(qc_reasons)
        return row

    try:
        symbolic = analyze_symbolic_dynamics(
            ppi_s,
            n_levels=SYMBOLIC_N_LEVELS,
        )
    except (TypeError, ValueError, RuntimeError) as exc:
        row["failure_stage"] = "symbolic_core"
        row["invalid_reason"] = str(exc)
        return row

    family_total = (
        symbolic.pct_0v
        + symbolic.pct_1v
        + symbolic.pct_2lv
        + symbolic.pct_2uv
    )
    invariant_checks = (
        symbolic.n_intervals == n_intervals,
        symbolic.n_words == symbolic.n_intervals - 2,
        np.isclose(family_total, 100.0),
        np.isclose(
            symbolic.pct_2v,
            symbolic.pct_2lv + symbolic.pct_2uv,
        ),
        np.isfinite([
            symbolic.pct_0v,
            symbolic.pct_1v,
            symbolic.pct_2lv,
            symbolic.pct_2uv,
            symbolic.pct_2v,
        ]).all(),
    )

    row.update({
        "n_words": symbolic.n_words,
        "pct_0v": symbolic.pct_0v,
        "pct_1v": symbolic.pct_1v,
        "pct_2lv": symbolic.pct_2lv,
        "pct_2uv": symbolic.pct_2uv,
        "pct_2v": symbolic.pct_2v,
        "symbolic_invariants_pass": bool(all(invariant_checks)),
    })
    row["symbolic_valid"] = row["symbolic_invariants_pass"]
    row["analysis_included"] = (
        row["ppi_qc_pass"] and row["symbolic_valid"]
    )

    if not row["symbolic_valid"]:
        row["failure_stage"] = "symbolic_validation"
        row["invalid_reason"] = "symbolic_invariant_failure"

    return row

## Session runner

`symbolic_analysis_run(session_id)` loads all 60/120/180-s processed windows, logs the complete input state, filters stationarity, analyzes every retained row, validates the output schema/invariants, writes atomically, reloads the CSV, and returns a compact run report.

In [6]:
def _validate_session_id(session_id: int) -> int:
    if isinstance(session_id, (bool, np.bool_)):
        raise TypeError("session_id must be a positive integer.")
    if not isinstance(session_id, Integral):
        raise TypeError("session_id must be a positive integer.")
    session_id = int(session_id)
    if session_id < 1:
        raise ValueError("session_id must be a positive integer.")
    return session_id


def _validate_output_frame(
    frame: pd.DataFrame,
    session_id: int,
) -> None:
    if frame.empty:
        raise ValueError("No stationary processed windows were analyzed.")
    if frame.columns.tolist() != OUTPUT_COLUMNS:
        raise ValueError("Output columns do not match the frozen schema.")
    if frame["window_uid"].duplicated().any():
        raise ValueError("Duplicate window_uid values detected.")
    if not (frame["session_id"] == session_id).all():
        raise ValueError("Output contains a different session_id.")
    if not (frame["representation"] == REPRESENTATION).all():
        raise ValueError("Output contains a non-processed representation.")
    if not frame["stationarity_pass_processed"].all():
        raise ValueError("A non-stationary window entered the output.")

    valid = frame["symbolic_valid"]
    if valid.any():
        metrics = frame.loc[
            valid,
            ["pct_0v", "pct_1v", "pct_2lv", "pct_2uv", "pct_2v"],
        ].to_numpy(dtype=float)
        if not np.isfinite(metrics).all():
            raise ValueError("Valid symbolic rows contain non-finite metrics.")

        family_total = metrics[:, :4].sum(axis=1)
        if not np.allclose(family_total, 100.0):
            raise ValueError("Symbolic family totals are not 100%.")
        if not np.allclose(
            metrics[:, 4],
            metrics[:, 2] + metrics[:, 3],
        ):
            raise ValueError("2V decomposition invariant failed.")

    expected_included = frame["ppi_qc_pass"] & frame["symbolic_valid"]
    if not np.array_equal(
        frame["analysis_included"].to_numpy(),
        expected_included.to_numpy(),
    ):
        raise ValueError("analysis_included policy mismatch.")


def symbolic_analysis_run(session_id: int) -> dict:
    """Run and save processed symbolic analysis for one session."""
    session_id = _validate_session_id(session_id)
    source_file = f"sample_{session_id}.csv"
    session_tag = f"session_{session_id:02d}"

    LOGGER.info("[%s] START", session_tag)
    LOGGER.info(
        "[%s][CONFIG] representation=%s windows=%s n_levels=%d",
        session_tag,
        REPRESENTATION,
        WINDOW_SIZES,
        SYMBOLIC_N_LEVELS,
    )

    try:
        batches, metadata = load_segmented_session(
            source_file,
            data_dir=SEGMENTED_DATA_DIR,
            window_sizes=WINDOW_SIZES,
            representation=REPRESENTATION,
            labels=None,
            stationarity_only=False,
        )
    except Exception:
        LOGGER.exception("[%s][INPUT] FAIL loading segmented data", session_tag)
        raise

    LOGGER.info(
        "[%s][INPUT] archive=%s",
        session_tag,
        metadata.attrs.get("archive_path", "unknown"),
    )

    stationary_windows = []
    total_windows = 0
    total_stationary = 0

    for window_size_s in WINDOW_SIZES:
        batch = batches[window_size_s]
        signals = np.asarray(batch["signal"], dtype=np.float64)
        labels = np.asarray(batch["label"], dtype=int)
        stationary = np.asarray(batch["stationarity_pass"], dtype=bool)
        fs = float(batch["fs"])

        if signals.ndim != 2:
            raise ValueError(
                f"{window_size_s}s signal batch must be two-dimensional."
            )
        if not np.isfinite(signals).all():
            raise ValueError(
                f"{window_size_s}s batch contains non-finite samples."
            )
        if not np.isfinite(fs) or fs <= 0:
            raise ValueError(f"Invalid sampling rate for {window_size_s}s.")
        if signals.shape[0] != labels.size or labels.size != stationary.size:
            raise ValueError(f"Misaligned arrays for {window_size_s}s.")

        n_total = int(labels.size)
        n_stationary = int(stationary.sum())
        n_excluded = n_total - n_stationary
        total_windows += n_total
        total_stationary += n_stationary

        pass_awake = int(np.sum(stationary & (labels == 0)))
        pass_drowsy = int(np.sum(stationary & (labels == 1)))
        LOGGER.info(
            (
                "[%s][INPUT][%ds] total=%d stationary_pass=%d "
                "excluded=%d pass_awake=%d pass_drowsy=%d"
            ),
            session_tag,
            window_size_s,
            n_total,
            n_stationary,
            n_excluded,
            pass_awake,
            pass_drowsy,
        )

        for array_index in np.flatnonzero(stationary):
            label_code = int(labels[array_index])
            if label_code not in STATE_NAMES:
                raise ValueError(
                    f"Unsupported label {label_code} in {window_size_s}s."
                )
            stationary_windows.append({
                "window_size_s": window_size_s,
                "window_id": int(batch["window_id"][array_index]),
                "source_row_index": int(batch["row_index"][array_index]),
                "state": STATE_NAMES[label_code],
                "label_code": label_code,
                "start_time_s": float(batch["start_time"][array_index]),
                "end_time_s": float(batch["end_time"][array_index]),
                "sampling_rate_hz": fs,
                "stationarity_score": float(
                    batch["stationarity_score"][array_index]
                ),
                "stationarity_pass": True,
                "signal": signals[array_index].copy(),
            })

    LOGGER.info(
        "[%s][INPUT] PASS total=%d stationary_input=%d excluded=%d",
        session_tag,
        total_windows,
        total_stationary,
        total_windows - total_stationary,
    )

    rows = []
    for window in stationary_windows:
        row = analyze_stationary_window(window, session_id)
        rows.append(row)
        if not row["analysis_included"]:
            LOGGER.warning(
                (
                    "[%s][WINDOW][%ds][W%03d] FAIL stage=%s reason=%s"
                ),
                session_tag,
                window["window_size_s"],
                window["window_id"],
                row["failure_stage"],
                row["invalid_reason"],
            )

    frame = (
        pd.DataFrame(rows)
        .loc[:, OUTPUT_COLUMNS]
        .sort_values(["window_size_s", "start_time_s", "window_id"])
        .reset_index(drop=True)
    )
    _validate_output_frame(frame, session_id)

    for window_size_s in WINDOW_SIZES:
        group = frame[frame["window_size_s"] == window_size_s]
        LOGGER.info(
            (
                "[%s][SYMBOLIC][%ds] rows=%d ppi_qc_pass=%d "
                "symbolic_valid=%d included=%d"
            ),
            session_tag,
            window_size_s,
            len(group),
            int(group["ppi_qc_pass"].sum()),
            int(group["symbolic_valid"].sum()),
            int(group["analysis_included"].sum()),
        )

    n_failed = int((~frame["analysis_included"]).sum())
    run_status = "PASS" if n_failed == 0 else "REVIEW_REQUIRED"

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / f"{session_tag}_symbolic_analysis.csv"
    temporary_path = output_path.with_suffix(".csv.tmp")
    frame.to_csv(temporary_path, index=False, na_rep="NaN")
    temporary_path.replace(output_path)

    reloaded = pd.read_csv(output_path)
    if reloaded.columns.tolist() != OUTPUT_COLUMNS:
        raise ValueError("Reloaded CSV schema mismatch.")
    if len(reloaded) != len(frame):
        raise ValueError("Reloaded CSV row-count mismatch.")
    if set(reloaded["window_uid"]) != set(frame["window_uid"]):
        raise ValueError("Reloaded CSV window_uid mismatch.")

    LOGGER.info(
        "[%s][OUTPUT] %s rows=%d bytes=%d",
        session_tag,
        output_path,
        len(frame),
        output_path.stat().st_size,
    )
    LOGGER.info(
        (
            "[%s][DONE] status=%s input_stationary=%d "
            "symbolic_valid=%d failed=%d"
        ),
        session_tag,
        run_status,
        total_stationary,
        int(frame["symbolic_valid"].sum()),
        n_failed,
    )

    return {
        "session_id": session_id,
        "status": run_status,
        "output_path": output_path,
        "n_segmented_windows": total_windows,
        "n_stationary_windows": total_stationary,
        "n_symbolic_valid": int(frame["symbolic_valid"].sum()),
        "n_failed": n_failed,
    }

## Run Session 1

Replace `SESSIONS = [1]` with the desired available session IDs when scaling. Each iteration writes one CSV before the next session begins.

In [7]:
SESSIONS = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 21, 22, 23, 25]
print(len(SESSIONS))

19


In [8]:
run_reports = []
for session_id in SESSIONS:
    run_reports.append(symbolic_analysis_run(session_id))



INFO | [session_04] START
INFO | [session_04][CONFIG] representation=processed windows=(60, 120, 180) n_levels=6
INFO | [session_04][INPUT] archive=/home/vutu0809/Desktop/NTSA_Foundation/phase1/segmentated_data/dhdata/sample_4.npz
INFO | [session_04][INPUT][60s] total=46 stationary_pass=46 excluded=0 pass_awake=29 pass_drowsy=17
INFO | [session_04][INPUT][120s] total=20 stationary_pass=20 excluded=0 pass_awake=13 pass_drowsy=7
INFO | [session_04][INPUT][180s] total=13 stationary_pass=13 excluded=0 pass_awake=8 pass_drowsy=5
INFO | [session_04][INPUT] PASS total=79 stationary_input=79 excluded=0
INFO | [session_04][SYMBOLIC][60s] rows=46 ppi_qc_pass=46 symbolic_valid=46 included=46
INFO | [session_04][SYMBOLIC][120s] rows=20 ppi_qc_pass=20 symbolic_valid=20 included=20
INFO | [session_04][SYMBOLIC][180s] rows=13 ppi_qc_pass=13 symbolic_valid=13 included=13
INFO | [session_04][OUTPUT] /home/vutu0809/Desktop/NTSA_Foundation/phase1/results/symbolic/processed/session_04_symbolic_analysis.cs